<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">

## Toxicity classification on molecular graphs using Graph Neural Networks (GNN) and `PyG`

**Goal** Gain hands-on experience with `PyG` in handling GNNs and `DataLoaders`. This notebook applies GNNs using `PyG` to classify molecular graphs from the Tox21 dataset. Each molecule is represented as a graph \( G = (V, E) \), where atoms are nodes and bonds are edges. Node and edge features are passed through message-passing layers to predict toxicity (binary classification).

</div>    

<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">

Assume we have a (molecular) graph $G = (V,E)$ and that we define features/embeddings to represent both nodes and edges, so that ${\mathbf{x}_i \in \mathbb{R}^{node_{in}}; \mathbf{e}_{ij} \in \mathbf{R}^{edge_{in}}}$. A key component of GNNs is **message passing**: each node aggregates information from its neighbors, either with or without modulation by the connecting bond (i.e., edge features).

Intuitively: $$\mathbf{x}_i^{(l+1)} = f \left(\mathbf{x}_i^{(l)}, \mbox{pooling}\left( \left\{\mbox{message}(\mathbf{x}_j^{(l)}, \mathbf{e}_{ij}) \right\}_{j \in N(i)} \right) \right)$$

Note that (here) the node embedding $\mathbf{x}_i$ (representation/featurization) is the only one being propagated, regardless of whether both node and/or edge features are used.

The `message` function is usually represented by a suitable (or, by a set of suitable) matrix $\mathbf{W}$, whose weights are trainable:
* if **node passing**: then, the type of edge between a node and its nearest neighbors does not matter; $\mathbf{W}_i = \mathbf{W}_i$
* if **edge attribute passing**, then, the type of edge modulates the information massing, and the matrix is unique per each bond type $\mathbf{W}_i = \mathbf{W}[i, e_{ij}]$

Here we define two GNN architectures:
- A **node-only GNN** using standard `GraphConv`, which **does not** support edge-conditioned message passing
- An **edge-aware GNN** using `NNConv`, which incorporates bond features into message passing


<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">
    
**Training dataset** : goal is to predict chemical toxicity using Graph Neural Networks

* ~7,800 molecules represented by SMILES strings, each with the outputs from 12 binary classification assays. Labels are either 1 = active, 0 = inactive or NaN = not tested
* Assays include nuclear receptor signaling pathways (7 assays) and stress response pathways (5 assays)
</div>

<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">
    
**Tools**
* `scikit-learn`, `torch`
* **Core cheminfo**: `RDKit` **fingerprints, descriptors** to automate molecular representation and generate the input to the classifier
* `PyG` to featurize the molecular graphs, define the GNN layers, the head of network is a classifier
* `torch` to train and test, using **batches** and **early_stopping**
</div>

In [ ]:
# this is to install the packages on Colab
##!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-2.0.1+cu118.html
#!pip install torch-geometric

#!pip install rdkit-pypi

In [ ]:
import pandas as pd
import numpy as np
import copy
import torch, os, joblib, sys
import torch.nn as nn
import torch.nn.functional as F    # activation functions
#from torch.info import summary

import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from rdkit import Chem, DataStructs
from rdkit.Chem import Draw, Descriptors, AllChem

from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool, GraphConv, NNConv

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import auc, roc_curve, confusion_matrix, ConfusionMatrixDisplay

# LB defined quantities, imported from src
from src.utils import atom_features, bond_features, graph_featurizer_pygeom
from src.utils import split_data
from src.utils import model_training_classifier, model_testing

In [ ]:
input_file = './data/tox21.csv'  ## this needs possibly edits to point to the input
task = 'NR-AR'
print(f'Classification task to choose = {task}')

# training parameters
n_batches = 25
n_epochs = 200
patience = 10

# do we want to use edgepassing
edge_passing = 'on'
print(f'Using GNN classifier with {edge_passing} edge passing!')

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
#### Read in **Tox21** dataset, pick one classification task and featurize (calculate molecular graph embeddings as `PyG` Data objects)
</div>

In [ ]:
# Tox21 da MoleculeNet, downloaded from the internet, sicne the original link/url comes with restrictions
df = pd.read_csv(input_file)

# Print columns
print(list(df.columns))  # print first few databse columns (SMILES + target)
print('Number of molecules in dataset = ' + str(df.shape[0]))
print('Number of assays available for classification tasks = ' + str(df.shape[1]-2))

if task not in list(df.columns):
    print('acthung! there is no such classification task in the input file')

In [ ]:
# ### Node and Edge Features Used (extracted via RDKit)
# Each atom is represented by a 5-dimensional feature vector consisting of:
#- Atomic number, Degree, Implicit valence, Formal charge, Aromaticity (binary)

#Each bond is represented by a 4-dimensional one-hot vector encoding bond type:
#- Single, Double, Triple, Aromatic#
#
#These feature vectors are computed using RDKit and passed as `x` (node features) and `edge_attr` (edge features) in PyTorch Geometric `Data` objects.

use_edge = 'on'
full_data = []

for k, smile in enumerate(df['smiles'].values):
  # check if that task label is present or not (some assays were inconclusive on some molecules)
  if np.isnan(df[task].values[k]) == False:
    full_data.append(graph_featurizer_pygeom(Chem.MolFromSmiles(smile), k, df[task].values[k], edge_attrib=edge_passing))

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
### Define train, test and validation datasets, all organized in batches

</div>

In [ ]:
print('Organize data into train, test and validation datasets, use batches...')

# split into train, test, validation set using pytorch functionalities
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

train_dataset, val_dataset, test_dataset = split_data(full_data, train_ratio, val_ratio)

# convert to torch-geometric Datasets, using batches
train_loader = DataLoader(train_dataset, batch_size=n_batches, shuffle=True) # batch size to be used, shuffle ON for training
test_loader  = DataLoader( test_dataset, batch_size=n_batches, shuffle=False)
val_loader   = DataLoader(  val_dataset, batch_size=n_batches, shuffle=False)

In [ ]:
n_samples = len(full_data)
print(f'Total nubmber of samples = {n_samples}')

node_dim = train_dataset[0].x.shape[1]
print('Embedding size of nodes = ' + str(node_dim))

if edge_passing == 'on':
  edge_dim = train_dataset[0].edge_attr.shape[1]
  print('Embedding size of nodes = ' + str(edge_dim))
else:
  print('No edge messaging')

print(f'Number of training samples = {len(train_dataset)}')

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
### Define the GNN Classifier for a graph-based classification; use layers available in `PyG`
* **node-passing** classifier: [`GraphConv` + '`Relu`] + $\ldots$ + `global pooling` + `classifier`
* **edge-passing** classifier: [`MLP` + `NNConv` + '`Relu`] + $\ldots$ + `global pooling` + `classifier`

Use edge features $e_{ij}$ in message passing

$$\mathbf{x}_i^{(l+1)} = \sigma \left( \sum_{j \in N(i)} \mathbf{W}_{e_{ij}} \cdot \mathbf{x}_j^{(l)} \right), \qquad  \mathbf{W}_{e_{ij}}  = MLP(e_{ij}) $$

One introduces an `MLP` (Multi Layer Perceptron) that maps each edge embedding $\mathbf{e}){ij}$ to a weight matrix $\mathbf{W}_{e_{ij}}$. These weights are learned during training (= matrix entries are learnable parameters), and used to transform incoming node messages before aggregation. The messages from neighboring nodes, modulated by the edge features, are then aggregated (`NNConv`) and activated (`Relu`) to return the new node embedding of that node $i$

**Why edge-aware message passing matters** In molecules, the type of bond (i.e., edge attribute) between atoms affects reactivity, polarity, and electronic distribution. Incorporating edge features allows the model to modulate message passing based on bond type — single, double, aromatic, etc.


</div>

In [ ]:
class GNNclassifier(nn.Module):

    """
    Graph-level binary classifier using GraphConv layers from PyG.

    This model performs:
    - Node-level message passing with two GraphConv layers
    - Nonlinear activation (ReLU) after each convolution
    - Global mean pooling to produce a graph-level embedding
    - A final linear layer to output a single logit per graph

    Parameters
    ----------
    node_in_dim : int
        Dimension of input node features.
    hidden_dim : int
        Dimension of hidden node embeddings.

    Forward Inputs
    --------------
    x : torch.Tensor
        Node feature matrix of shape [num_nodes, node_in_dim].
    edge_index : torch.Tensor
        Edge indices in COO format of shape [2, num_edges].
    batch : torch.Tensor
        Batch vector mapping each node to its corresponding graph, of shape [num_nodes].

    Returns
    -------
    x : torch.Tensor
        Tensor of shape [batch_size, 1], containing logits for each graph in the batch.

    Notes
    -----
    - Output logits should be passed through a sigmoid during evaluation or loss computation.
    - No edge features are used in this model.
    """

    def __init__(self, node_in_dim, hidden_dim):
        super().__init__()

        self.conv1 = GraphConv(node_in_dim, hidden_dim)     # this is aready a GNN layer that performs message passing with the nearest neighobrs
        self.conv2 = GraphConv(hidden_dim, hidden_dim)
        self.classifier = torch.nn.Linear(hidden_dim, 1)  # single Linear, we can always make this more complex

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index)    # the linear layer is already implemented
        x = F.relu(x) # just need to apply a non linear activation function
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = global_mean_pool(x, batch)  # graph embedding, averaging over all the noves
        x = self.classifier(x)     # logits per graph, shape [batch_size, 1]
        return x

In [ ]:
class GNNclassifierEdgeAware(nn.Module):

    """
    Graph-level binary classifier using NNConv layers from PyG to handle edge message passing.

    This model performs:
    - Node-level AND edge-lebel message passing with two NNConv layers
    - Nonlinear activation (ReLU) after each convolution
    - Global mean pooling to produce a graph-level embedding
    - A final linear layer to output a single logit per graph

    Parameters
    ----------
    node_in_dim : int
        Dimension of input node features.
    hidden_dim : int
        Dimension of hidden node embeddings.
    edge_in_dim : int
        Dimension of input edge features.

    Forward Inputs
    --------------
    x : torch.Tensor
        Node feature matrix of shape [num_nodes, node_in_dim].
    edge_index : torch.Tensor
        Edge indices in COO format of shape [2, num_edges].
    edge_attrib : torch.Tensor
        Edge feature matrix of shape [num_edges, edge_in_dim].
    batch : torch.Tensor
        Batch vector mapping each node to its corresponding graph, of shape [num_nodes].

    Returns
    -------
    x : torch.Tensor
        Tensor of shape [batch_size, 1], containing logits for each graph in the batch.

    Notes
    -----
    - NNConv layers use learned MLPs (`edge_nn1`, `edge_nn2`) to transform edge attributes
      into dynamic weight matrices for message passing.
    - The classifier outputs a single logit per graph, which should be passed through `torch.sigmoid`
      during evaluation or loss computation.
    - Aggregation scheme is `'mean'`; this can be replaced with `'add'` or `'max'` as needed.
    """

    def __init__(self, node_in_dim, hidden_dim, edge_in_dim):
        super().__init__()

        # define MLP to generate weight matrices from edge features (can be any MPL layer): these return a flattened weight matrix of size
        # (in_channels, out_channels)
        self.edge_nn1 = nn.Sequential(
            nn.Linear(edge_in_dim, hidden_dim * node_in_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim * node_in_dim, hidden_dim * node_in_dim)
        )

        self.edge_nn2 = nn.Sequential(
            nn.Linear(edge_in_dim, hidden_dim * hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim * hidden_dim, hidden_dim * hidden_dim)
        )

        # perform message passing using both node and edge features
        self.conv1 = NNConv(in_channels = node_in_dim, out_channels = hidden_dim, nn = self.edge_nn1, aggr = 'mean')
        self.conv2 = NNConv(in_channels = hidden_dim, out_channels = hidden_dim, nn = self.edge_nn2, aggr = 'mean')

        # binary classification
        self.classifier = torch.nn.Linear(hidden_dim, 1)  # single Linear, we can always make this more complex

    def forward(self, x, edge_index, edge_attrib, batch):
        x = self.conv1(x, edge_index, edge_attrib)    # the linear layer is already implemented
        x = F.relu(x) # just need to apply a non linear activation function
        x = self.conv2(x, edge_index, edge_attrib)
        x = F.relu(x)
        x = global_mean_pool(x, batch)  # graph embedding, averaging over all the noves
        x = self.classifier(x)     # logits per graph, shape [batch_size, 1]
        return x

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
### Instantiate model, optimizer and Binary Cross Entropy (w Logits) loss function; then train the model using early stopping

</div>

In [ ]:
# instantiate model
if edge_passing == 'off':
    model = GNNclassifier(node_in_dim = node_dim, hidden_dim=8).to(device)  # in_dim is hard_coded, we might automate this based on the features
else:
    model = GNNclassifierEdgeAware(node_in_dim = node_dim, hidden_dim = 8, edge_in_dim = edge_dim).to(device)

print(model)  # print the architecture in terms of layers and input/output sizes

# define optimizer and loss function
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
loss_fn = nn.BCEWithLogitsLoss()

model = model_training_classifier(model, optimizer, loss_fn, train_loader, val_loader, n_epochs, device, patience, early_stop = 'on')

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
### Test trained GNN model on the test set molecular graphs, compute metrics
</div>

In [ ]:
# Set model in evaluation mode
y_pred, y_true = model_testing(model, test_loader, device, classifier = 'on')

print('Evaluation metrics for the best-fitting classificator:')
# compute a number of metrics that are then featured in the confusion matrix
print(" Accuracy: {:.2f}".format(accuracy_score(y_true, y_pred)))
print("Precision: {:.2f}".format(precision_score(y_true, y_pred)))
print("   Recall: {:.2f}".format(recall_score(y_true, y_pred)))
print(" F1 Score: {:.2f}".format(f1_score(y_true, y_pred)))

# add ROC AUC and confusion matrix!
# Plot confusion matrix
print('Plot confusion matrix relating to the classification...')
fig = plt.figure(figsize = (4,4))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred)).plot(cmap='Blues')
#plt.savefig(os.path.join('plots', 'Confusion_matrix_' + regres_model + '.pdf'), bbox_inches = 'tight')

# add ablation cell, where node features are removed

<div style="background-color: lightyellow; padding: 10px; border-radius: 5px;">

## Summary

In this notebook, we used Graph Neural Networks (GNNs) to classify molecular toxicity from the Tox21 dataset using `PyG`.

- We tested both **GraphConv** (node-only message passing) and **NNConv** (edge-aware message passing).
- Each molecule was represented as a graph, with atoms as nodes and bonds as edges. Features were extracted using `RDKit`, featurization is learned by the network from the graph structure
- Our best model reached **XXX AUC** and **0.97 accuracy** on the NR-AR classification task.
- This notebook demonstrates a scalable GNN pipeline for molecular property prediction, with batching, GPU usage, and featurization utilities.

Future improvements could include:
- Hyperparameter optimization
- Scaffold-based data splits
- Visualizing learned graph embeddings

</div>